# Pointwise LightGBM + SVD Standalone Evaluation

Binary LightGBM model predicting `booking_bool`. It complements LambdaRank by scoring each hotel-search row independently. The notebook also evaluates SVD latent factors as a standalone ranking signal.

In [1]:
import sys
from pathlib import Path

cwd = Path(".").resolve()
REPO_ROOT = cwd if (cwd / ".git").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

import json
from itertools import product

import numpy as np
import pandas as pd
import lightgbm as lgb

from src.utils import load_train, load_train_cached_sample, train_val_split, compute_ndcg_fast
from src.features import build_features
from src.svd_features import build_svd_features
from src.position_features import build_position_features
from src.modeling import get_lgb_device_params

In [2]:
USE_SAMPLE = False
SAMPLE_FRAC = 0.15
REFRESH_CACHE = False
DEVICE_PARAMS = get_lgb_device_params()

GPU mode enabled - using LightGBM GPU acceleration


## Load Data + Feature Engineering

In [3]:
if USE_SAMPLE:
    df = load_train_cached_sample(sample_frac=SAMPLE_FRAC, refresh=REFRESH_CACHE)
    print(f"Loaded {SAMPLE_FRAC*100:.0f}% sample")
else:
    df = load_train()
    print("Loaded full training data")

n_raw = df.shape[1]
df = build_features(df)
print(f"Loaded {len(df):,} rows, {df['srch_id'].nunique():,} searches")
print(f"Columns: {n_raw} raw + {df.shape[1] - n_raw} base features = {df.shape[1]} total")

Loaded full training data


/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["comp_cheaper_frac"] = (
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_missing"] = df[col].isnull().astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["month"] = dt.dt.month.astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:115: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["day_of_week"] = dt.dt.dayofweek.astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti

Loaded 4,958,347 rows, 199,795 searches
Columns: 54 raw + 64 base features = 118 total


/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["booking_window_bucket"] = pd.cut(
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:135: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["score2ma"] = (
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  C

## Train/Val Split + SVD Features

In [4]:
train_df, val_df = train_val_split(df)
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}")

# Build target-derived features from the training fold only (leakage-safe)
n_before = df.shape[1]
df = build_svd_features(train_df, df)
print(f"SVD features added: {df.shape[1] - n_before} columns")

n_before = df.shape[1]
df = build_position_features(train_df, df)
print(f"Position-debiased features added: {df.shape[1] - n_before} columns")

# Re-split to get augmented train/val (same random_state=42 gives same split)
train_df, val_df = train_val_split(df)

Train: 3,966,682  Val: 991,665


SVD features added: 43 columns


Position-debiased features added: 6 columns


## Prepare Features and Target

In [5]:
# Target: binary booking label
y_train = train_df["booking_bool"].values
y_val = val_df["booking_bool"].values

# Same exclusions as lambdarank notebook
DROP_COLS = [
    "srch_id", "date_time", "position", "click_bool", "booking_bool",
    "gross_bookings_usd", "random_bool",
]
COMP_RAW = [f"comp{i}_{s}" for i in range(1, 9) for s in ("rate", "inv", "rate_percent_diff")]
drop = set(DROP_COLS + COMP_RAW)
feature_cols = [c for c in train_df.columns if c not in drop]

print(f"Features: {len(feature_cols)}")

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

Features: 136


## Initial Training

In [6]:
params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "num_leaves": 127,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "min_child_samples": 50,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
    **DEVICE_PARAMS,
}

# Pointwise training treats rows independently
train_set = lgb.Dataset(X_train, label=y_train)
val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

model = lgb.train(
    params,
    train_set,
    valid_sets=[val_set],
    num_boost_round=4000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

Training until validation scores don't improve for 50 rounds


[100]	valid_0's binary_logloss: 0.110659


[200]	valid_0's binary_logloss: 0.109622


[300]	valid_0's binary_logloss: 0.109307


[400]	valid_0's binary_logloss: 0.109156


...

[800]	valid_0's binary_logloss: 0.10892


[900]	valid_0's binary_logloss: 0.108867


[1000]	valid_0's binary_logloss: 0.108837


[1100]	valid_0's binary_logloss: 0.10881


Early stopping, best iteration is:
[1085]	valid_0's binary_logloss: 0.108802


## Evaluation

In [7]:
val_df_eval = val_df.copy()
val_df_eval["pred_score"] = model.predict(X_val)

baseline_ndcg = compute_ndcg_fast(val_df_eval, "prop_location_score1")
pointwise_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")

print(f"Baseline NDCG@5 (location_score1): {baseline_ndcg:.4f}")
print(f"Pointwise LightGBM NDCG@5:         {pointwise_ndcg:.4f}")
print(f"LambdaRank NDCG@5:                 0.3820")
print(f"Improvement over baseline:          +{pointwise_ndcg - baseline_ndcg:.4f}")

Baseline NDCG@5 (location_score1): 0.1727
Pointwise LightGBM NDCG@5:         0.3909
LambdaRank NDCG@5:                 0.3820
Improvement over baseline:          +0.2183


## Feature Importance

In [8]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("Top 20 features by gain (pointwise):")
print(importance.head(20).to_string(index=False))

# Compare with LambdaRank if available
lambdarank_results = REPO_ROOT / "outputs" / "lambdarank_run1.json"
if lambdarank_results.exists():
    with open(lambdarank_results) as f:
        lr_data = json.load(f)
    if "top_features" in lr_data:
        print("\n\nLambdaRank top features (for comparison):")
        for feat, gain in list(lr_data["top_features"].items())[:10]:
            print(f"  {feat}: {gain:.1f}")

Top 20 features by gain (pointwise):
                       feature          gain
     prop_location_score2_rank 192140.458916
          prop_location_score2 168560.319771
            search_hotel_count 156652.782423
           dest_hotel_affinity 122984.032076
         price_usd_diff_median  91018.317762
                      score1d2  90184.775015
                price_usd_rank  85998.110301
    hotel_book_rate_by_country  84086.830158
          prop_starrating_rank  77440.174046
     prop_starrating_diff_mean  75179.457913
           price_vs_historical  64450.166201
           price_usd_diff_mean  63481.097938
            hotel_ctr_unbiased  61646.780768
        prop_review_score_rank  54222.708758
prop_location_score2_diff_mean  44266.490712
     orig_destination_distance  43410.477042
           srch_booking_window  43030.486704
                     price_usd  42340.277972
      prop_location_score2_std  41646.140169
   prop_review_score_diff_mean  41031.370875


LambdaRank top f

## Hyperparameter Tuning

Same grid as LambdaRank. Evaluate by **NDCG@5** (competition metric), not logloss.

In [9]:
search_space = {
    "num_leaves": [63, 127, 255],
    "learning_rate": [0.01, 0.05, 0.1],
    "min_child_samples": [20, 50, 100],
}

results = []
base_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
    **DEVICE_PARAMS,
}

keys = list(search_space.keys())
for combo in product(*search_space.values()):
    trial_params = {**base_params, **dict(zip(keys, combo))}

    train_set_hp = lgb.Dataset(X_train, label=y_train)
    val_set_hp = lgb.Dataset(X_val, label=y_val, reference=train_set_hp)

    m = lgb.train(
        trial_params,
        train_set_hp,
        valid_sets=[val_set_hp],
        num_boost_round=4000,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )

    preds = m.predict(X_val)
    val_df_eval["pred_score"] = preds
    score = compute_ndcg_fast(val_df_eval, "pred_score")

    row = dict(zip(keys, combo))
    row["ndcg5"] = score
    row["n_trees"] = m.best_iteration
    results.append(row)
    print(f"  leaves={row['num_leaves']:>3}  lr={row['learning_rate']:.2f}  "
          f"min_child={row['min_child_samples']:>3}  -> NDCG@5={score:.4f}  trees={m.best_iteration}")

results_df = pd.DataFrame(results).sort_values("ndcg5", ascending=False)
print("\nTop 5 configurations:")
print(results_df.head().to_string(index=False))

  leaves= 63  lr=0.01  min_child= 20  -> NDCG@5=0.3909  trees=4000


  leaves= 63  lr=0.01  min_child= 50  -> NDCG@5=0.3913  trees=4000


  leaves= 63  lr=0.01  min_child=100  -> NDCG@5=0.3918  trees=4000


  leaves= 63  lr=0.05  min_child= 20  -> NDCG@5=0.3861  trees=670


  leaves= 63  lr=0.05  min_child= 50  -> NDCG@5=0.3922  trees=1939


...

  leaves=255  lr=0.05  min_child= 50  -> NDCG@5=0.3896  trees=463


  leaves=255  lr=0.05  min_child=100  -> NDCG@5=0.3931  trees=705


  leaves=255  lr=0.10  min_child= 20  -> NDCG@5=0.3792  trees=97


  leaves=255  lr=0.10  min_child= 50  -> NDCG@5=0.3838  trees=137


  leaves=255  lr=0.10  min_child=100  -> NDCG@5=0.3856  trees=194

Top 5 configurations:
 num_leaves  learning_rate  min_child_samples    ndcg5  n_trees
        255           0.01                100 0.398636     3968
        255           0.01                 20 0.397346     4000
        255           0.01                 50 0.397226     3971
        127           0.01                100 0.396515     4000
        127           0.01                 50 0.394705     4000


## Retrain with Best Params

In [10]:
best = results_df.iloc[0]
print(f"Best config: leaves={int(best.num_leaves)}, lr={best.learning_rate}, "
      f"min_child={int(best.min_child_samples)}, NDCG@5={best.ndcg5:.4f}")

best_params = {
    **base_params,
    "num_leaves": int(best.num_leaves),
    "learning_rate": best.learning_rate,
    "min_child_samples": int(best.min_child_samples),
}

train_set_best = lgb.Dataset(X_train, label=y_train)
val_set_best = lgb.Dataset(X_val, label=y_val, reference=train_set_best)

best_model = lgb.train(
    best_params,
    train_set_best,
    valid_sets=[val_set_best],
    num_boost_round=4000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

val_df_eval["pred_score"] = best_model.predict(X_val)
final_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")
print(f"\nFinal Pointwise NDCG@5: {final_ndcg:.4f}")

# Save model
models_dir = REPO_ROOT / "models"
models_dir.mkdir(exist_ok=True)
best_model.save_model(str(models_dir / "pointwise_best.txt"))

Best config: leaves=255, lr=0.01, min_child=100, NDCG@5=0.3986


Training until validation scores don't improve for 50 rounds


[100]	valid_0's binary_logloss: 0.114854


[200]	valid_0's binary_logloss: 0.112162


[300]	valid_0's binary_logloss: 0.111034


...

[3700]	valid_0's binary_logloss: 0.107951


[3800]	valid_0's binary_logloss: 0.107939


[3900]	valid_0's binary_logloss: 0.107925


[4000]	valid_0's binary_logloss: 0.107914
Did not meet early stopping. Best iteration is:
[3998]	valid_0's binary_logloss: 0.107913



Final Pointwise NDCG@5: 0.3982


## Save Results

In [11]:
out_dir = REPO_ROOT / "outputs"
out_dir.mkdir(exist_ok=True)

# Results JSON
results_out = {
    "model": "pointwise",
    "use_sample": USE_SAMPLE,
    "sample_frac": SAMPLE_FRAC if USE_SAMPLE else None,
    "n_rows": len(df),
    "n_searches": int(df["srch_id"].nunique()),
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_features": len(feature_cols),
    "n_trees": int(best_model.best_iteration),
    "best_params": {k: v for k, v in best_params.items()
                    if k not in ("verbose", "num_threads", "seed")},
    "baseline_ndcg5": round(baseline_ndcg, 4),
    "model_ndcg5": round(final_ndcg, 4),
    "top_features": {
        row["feature"]: round(row["gain"], 1)
        for _, row in importance.head(15).iterrows()
    },
}

with open(out_dir / "pointwise_run1.json", "w") as f:
    json.dump(results_out, f, indent=2)
print("Results saved to outputs/pointwise_run1.json")

# Save val predictions for ensemble
val_preds = val_df_eval[["srch_id", "prop_id", "booking_bool", "click_bool", "pred_score"]].copy()
val_preds.to_parquet(out_dir / "pointwise_val_preds.parquet", index=False)
print(f"Validation predictions saved: {len(val_preds):,} rows")

Results saved to outputs/pointwise_run1.json
Validation predictions saved: 991,665 rows


## SVD Standalone Evaluation

Evaluate SVD affinity as a standalone recommender-style ranking signal.

In [12]:
ndcg_svd = compute_ndcg_fast(val_df_eval, "svd_affinity_score")

print("=" * 50)
print("Model Comparison (NDCG@5)")
print("=" * 50)
print(f"  Baseline (location_score1):  {baseline_ndcg:.4f}")
print(f"  SVD standalone:              {ndcg_svd:.4f}")
print(f"  Pointwise LightGBM:          {final_ndcg:.4f}")
print(f"  LambdaRank:                  0.3820")
print("=" * 50)

# Save SVD standalone results
svd_out = {
    "model": "svd_standalone",
    "n_components": 20,
    "ndcg5": round(ndcg_svd, 4),
    "baseline_ndcg5": round(baseline_ndcg, 4),
}
with open(out_dir / "svd_standalone.json", "w") as f:
    json.dump(svd_out, f, indent=2)
print(f"\nSVD standalone results saved to outputs/svd_standalone.json")

Model Comparison (NDCG@5)
  Baseline (location_score1):  0.1727
  SVD standalone:              0.2196
  Pointwise LightGBM:          0.3982
  LambdaRank:                  0.3820

SVD standalone results saved to outputs/svd_standalone.json


### SVD Standalone Interpretation

Collaborative filtering is a weak standalone fit for this dataset because the interactions are sparse and session-based. Each search is anonymous, so there are no persistent user histories to build a dense user-item matrix.

The SVD signal still has value as a feature source: `svd_affinity_score` and the latent factors give the tree models destination-hotel affinity information that can be combined with price, location, and search context.